# Agentic AI to help us to make final judgement
In this notebook, we will one of the startegy (similar to `2_backtesting_py_exp`) and after backtesting; we will use it for read trading. Our final decision will always be made by LLM agent.

Install ta-lib on Mac with brew first `brew install ta-lib` and then `pip install TA-Lib`

In [1]:
import pandas as pd
import pandas_ta as ta
import numpy as np
from backtesting import Backtest, Strategy
from backtesting.lib import crossover
from backtesting.test import SMA, GOOG
import yfinance as yf
import talib 
from utils import get_stock_data
import time
import warnings

/Users/ikespand/anaconda3/envs/py311/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/Users/ikespand/anaconda3/envs/py311/lib/python3.11/site-packages/backtesting/_plotting.py:55: UserWarning: Jupyter Notebook detected. Setting Bokeh output to notebook. This may not work in Jupyter clients without JavaScript support, such as old IDEs. Reset with `backtesting.set_bokeh_output(notebook=False)`.
  warnings.warn('Jupyter Notebook detected. '


Loading BokehJS ...

## Typical TA
Using from `ta-lib` library.

In [47]:
class DonchianBreakoutStrategy(Strategy):
    breakout_period = 20
    exit_period = 10
    atr_period = 14
    atr_mult = 2.5

    def init(self):
        df = self.data.df
        self.highest_high = self.I(lambda: df['High'].rolling(self.breakout_period).max())
        self.lowest_low = self.I(lambda: df['Low'].rolling(self.exit_period).min())
        self.atr = self.I(lambda: df.ta.atr(length=self.atr_period))

    def next(self):
        price = self.data.Close[-1]

        # Entry: breakout above 20-day high
        if not self.position and price > self.highest_high[-2]:
            sl = price - self.atr[-1] * self.atr_mult
            self.buy(sl=sl)

        # Exit: breakout below 10-day low or hit stop
        elif self.position and price < self.lowest_low[-2]:
            self.position.close()

# Fetch data
df = get_stock_data("BTC-USD", interval="1d", start="2024-02-01", end="2025-04-20")

# Run backtest
bt = Backtest(df, DonchianBreakoutStrategy, cash=1000_000, commission=0.002)
stats = bt.run()

print(stats)
# bt.plot(filename="DonchianBreakoutStrategy.html")

Start                     2024-02-01 00:00:00
End                       2025-04-19 00:00:00
Duration                    443 days 00:00:00
Exposure Time [%]                    46.84685
Equity Final [$]                1324697.96476
Equity Peak [$]                 1437284.80621
Commissions [$]                   24823.11727
Return [%]                            32.4698
Buy & Hold Return [%]                 62.6922
Return (Ann.) [%]                    26.00532
Volatility (Ann.) [%]                43.64646
CAGR [%]                             26.07108
Sharpe Ratio                          0.59582
Sortino Ratio                         1.23357
Calmar Ratio                          0.84253
Alpha [%]                             7.01357
Beta                                  0.40605
Max. Drawdown [%]                   -30.86567
Avg. Drawdown [%]                    -7.96776
Max. Drawdown Duration      253 days 00:00:00
Avg. Drawdown Duration       46 days 00:00:00
# Trades                          

## Integrating ESN model from `1a_esn_le2e.ipynb`
I further experimented with ESN in `1a_esn_le2e.ipynb` and want to use this only for future. So backtesting:

In [28]:
from utils import StockPricePredictor

predictor = StockPricePredictor(model_path="esnmodel_BTC-USD_4_1d_2015-01-01_2024-12-31.pkl", 
                                scaler_path="scaler_BTC-USD_4_1d_2015-01-01_2024-12-31.pkl")


def get_trade_signal_with_esn(df, buy_threshold=0.02, short_threshold=0.09):
    """
    Determines if a BUY or SHORT signal should be generated based on ESN model prediction.
    
    Args:
        df (DataFrame): The dataframe containing the latest stock data.
        buy_threshold (float): % increase to trigger BUY.
        short_threshold (float): % decrease to trigger SHORT.
    
    Returns:
        str: "BUY", "SHORT", or "HOLD"
    """
    predicted_close = predictor.predict(df)
    last_close = df.iloc[-1]['Close']
    
    if predicted_close >= last_close * (1 + buy_threshold):
        return "BUY"
    elif predicted_close <= last_close * (1 - short_threshold):
        return "SHORT"
    return "HOLD"



class MLTradingStrategy(Strategy):
    feature_window = 4 # window size for features i.e., hisorical data length
    stop_loss_pct = 0.02
    take_profit_pct = 0.04

    def init(self):
        pass

    def next(self):
        N = self.feature_window  
        # Ensure enough data to generate features
        if len(self.data.Close) < N:
            return 

        # Extract last `N` rows into a DataFrame
        df = pd.DataFrame({
            "Open": self.data.Open[-N:],
            "High": self.data.High[-N:],
            "Low": self.data.Low[-N:],
            "Close": self.data.Close[-N:],
            #"Volume": self.data.Volume[-N:],
        })

        # Pass last `N` rows to ML model
        signal = get_trade_signal_with_esn(df)  
        price = self.data.Close[-1]

        if signal == "BUY":
            sl = price * (1 - self.stop_loss_pct)
            tp = price * (1 + self.take_profit_pct)
            self.buy(sl=sl, tp=tp)

        elif signal == "SHORT":
            sl = price * (1 + self.stop_loss_pct)
            tp = price * (1 - self.take_profit_pct)
            self.sell(sl=sl, tp=tp)

df = get_stock_data("BTC-USD", interval="1d", start="2024-02-01", end="2025-04-20")

# Run Backtest
bt = Backtest(df, MLTradingStrategy, cash=1000_000, commission=0.002)

stats = bt.run()

# Print Results
print(stats)
bt.plot(filename="MLTradingStrategy.html")

Start                     2024-02-01 00:00:00
End                       2025-04-19 00:00:00
Duration                    443 days 00:00:00
Exposure Time [%]                     0.67568
Equity Final [$]                 1035318.2279
Equity Peak [$]                  1035318.2279
Commissions [$]                    3995.76913
Return [%]                            3.53182
Buy & Hold Return [%]                97.47391
Return (Ann.) [%]                     2.89441
Volatility (Ann.) [%]                 2.53827
CAGR [%]                              2.90104
Sharpe Ratio                          1.14031
Sortino Ratio                       337.45747
Calmar Ratio                        306.31172
Alpha [%]                             3.31841
Beta                                  0.00219
Max. Drawdown [%]                    -0.00945
Avg. Drawdown [%]                    -0.00945
Max. Drawdown Duration        2 days 00:00:00
Avg. Drawdown Duration        2 days 00:00:00
# Trades                          

GridPlot(id='p1327', ...)

## Using LLM agent to decide for future (live trading)
Here, I use crewai and OpenAI's model for the LLM. My OpenAI key is store in `.env` file.

In [ ]:
from dotenv import load_dotenv
import os
from crewai import Crew, Agent, Task
from langchain_openai import ChatOpenAI
import json
from langchain_core.caches import BaseCache
from langchain_core.caches import InMemoryCache
from datetime import datetime
from crewai_tools import ScrapeWebsiteTool
from agentic_tools import WebSearchTool
import re
from pprint import pprint 

load_dotenv()
try:
    os.environ["OPENAI_API_KEY"]
except KeyError:
    print("Please provide OPEN_API_KEY as env variable!")


/Users/ikespand/anaconda3/envs/py311/lib/python3.11/site-packages/pydantic/_internal/_generate_schema.py:623: UserWarning: <built-in function callable> is not a Python type (it may be an instance of an object), Pydantic will allow any object with no validation since we cannot even enforce that the input is an instance of the given type. To get rid of this error wrap the type with `pydantic.SkipValidation`.
  warn(


In [ ]:
BaseCache.global_cache = InMemoryCache()
ChatOpenAI.model_rebuild()
llm = ChatOpenAI(model="gpt-4o")

# Add 2 tools for agents
search_tool = WebSearchTool()
scrape_tool = ScrapeWebsiteTool()

# Agent-1: News Analyst Agent
news_analyst = Agent(
    role="Market Sentiment Analyst",
    goal="Analyze recent news and public sentiment to determine external factors affecting {ticker}, including macroeconomic, geopolitical, and company-specific events.",
    backstory="An experienced financial journalist and sentiment analyst skilled in identifying impactful stories and gauging market tone from various news and media sources.",
    tools=[search_tool, scrape_tool],
    verbose=True,
    allow_delegation=False,
    llm=llm
)

# Agent-2: Trading Analyst Agent
trading_analyst = Agent(
    role="Quantitative & Fundamental Trading Strategist",
    goal="Generate trading decisions by integrating ML-based price predictions, technical backtests, and real-world sentiment factors for a holistic view of the stock's opportunity.",
    backstory="A market strategist with deep expertise in ML-driven signals, technical trading systems, and interpreting real-world sentiment shifts to make balanced decisions with clear risk management.",
    llm=llm,
    allow_delegation=False
)



def create_trading_crew(ticker, current_price, predicted_price, backtesting_results):
    today = datetime.now().strftime("%B %d, %Y")  

    # Task 1: Sentiment & News Impact
    sentiment_task = Task(
        description=f"""
        Research and summarize the **current market sentiment** and any **external events** that may influence trading for {ticker}.

        Include:
        - News and media sentiment (positive/negative/neutral)
        - Macroeconomic factors (e.g., inflation, interest rates, Fed decisions)
        - Industry trends (e.g., crypto regulations for BTC-USD)
        - Company-specific news (if applicable)
        - Source credibility (mention top outlets like Bloomberg, CNBC, Reuters, etc.)

        Focus on the most recent 7–14 days as of {today}. Avoid outdated info.

        Output:
        - A brief summary of tone and its potential market impact.
        - A categorized list of influencing factors (macro, industry, company).
        - URLs to key sources.
        """,
        expected_output="Sentiment tone + categorized market influences with source links.",
        agent=news_analyst
    )

    # Task 2: Final Trading Decision
    trading_task = Task(
        description=f"""
        Based on the full context of {ticker} on {today}, make a smart trading decision.

        You have access to:
        1. **Current Market Price:** {current_price}
        2. **Predicted Price (from ML):** {predicted_price}
        3. **Backtesting Results:** {backtesting_results}
        4. **Sentiment Summary & Influencing Factors** (from the Market Sentiment Analyst)

        Assess all data together and provide:
        - A clear action: "BUY", "SELL", or "HOLD"
        - Stop-loss and take-profit levels based on volatility or past performance
        - Short explanation with reference to technical (ML/backtest) and sentiment (news) justification

        Return JSON:
        {{
        "action": "BUY" or "SELL" or "HOLD",
        "stop_loss": number,
        "take_profit": number,
        "reasoning": "brief summary of how all factors influence this decision"
        }}
        """,
        expected_output='''{"action": "BUY" or "SELL" or "HOLD", "stop_loss": number, "take_profit": number, "reasoning": "string"}''',
        agent=trading_analyst
    )


    crew = Crew(
        agents=[news_analyst, trading_analyst],
        tasks=[sentiment_task, trading_task],
        verbose=False
    )

    return crew.kickoff()


In [43]:
df_latest = get_stock_data("BTC-USD", interval="1d", start="2025-04-01", end="2025-04-20")
df_latest.drop(["Volume"], axis=1, inplace=True)
predicted_close_today = predictor.predict(df_latest.tail(4))
is_buy_signal = get_trade_signal_with_esn(df_latest.tail(4))
print("predicted_close_today", predicted_close_today)
print("current_price (/last closing price)", df_latest.iloc[-1]['Close'])
print("signal_from_ml", is_buy_signal)

predicted_close_today 83853.93884705503
current_price (/last closing price) 85063.4140625
signal_from_ml HOLD


In [ ]:
# Create and run the crew
result = create_trading_crew(
    ticker="BTC-USD",
    current_price=df_latest.iloc[-1]['Close'],
    predicted_price=predicted_close_today,
    backtesting_results=stats
)
print("\nFinal Recommendation:\n", result)

# Agent: Market Sentiment Analyst
## Task: 
        Research and summarize the **current market sentiment** and any **external events** that may influence trading for BTC-USD.

        Include:
        - News and media sentiment (positive/negative/neutral)
        - Macroeconomic factors (e.g., inflation, interest rates, Fed decisions)
        - Industry trends (e.g., crypto regulations for BTC-USD)
        - Company-specific news (if applicable)
        - Source credibility (mention top outlets like Bloomberg, CNBC, Reuters, etc.)

        Focus on the most recent 7–14 days as of April 20, 2025. Avoid outdated info.

        Output:
        - A brief summary of tone and its potential market impact.
        - A categorized list of influencing factors (macro, industry, company).
        - URLs to key sources.
        


# Agent: Market Sentiment Analyst
## Thought: Thought: To provide an accurate analysis, I need to search for recent news articles and reports about BTC-USD, focusing on 

In [ ]:
match = re.search(r"json\s*(\{.*?\})\s*", str(result), re.DOTALL) 
if match: 
    json_str = match.group(1)
    agentic_trade_decision = json.loads(json_str)
    pprint(agentic_trade_decision)
else:
    print("Something went wrong in decoding/formatting returned string as JSON.")

{'action': 'HOLD',
 'reasoning': "The decision to 'HOLD' is based on a mixed sentiment with "
              'bullish potential offset by bearish market indicators. The '
              'current market price ($85,063) is slightly above the predicted '
              'ML model price ($83,854) indicating a potential minor '
              'correction. However, sentiment suggests a potential surge '
              'towards $90,000-$95,000, driven by market optimism. The '
              'backtesting results show stable performance but underperform '
              'relative to buy & hold strategy, suggesting limited upside '
              'under current volatility. The Crypto Fear and Greed Index '
              'reflects ongoing market fear which could lead to price '
              'declines. Therefore, setting a stop-loss at $83,000 to manage '
              'downside risk and a take-profit at $90,000 to capture potential '
              'upside aligns with current volatility and sentiment inf

## 3. Using paper trading
Here, I use Alpaca's paper trading API to execute a trade with TP and SL. We can then see how it performs. I have created a free account at Alpaca and saved keys to same `.env` file. Here is the link: https://app.alpaca.markets/paper/dashboard/overview

In [ ]:
import alpaca_trade_api as tradeapi
load_dotenv()

try:
    os.environ["ALPACA_API_KEY"]
    os.environ["ALPACA_API_SECRET"]
except KeyError:
    print("Please provide ALPACA_API_KEY and/or ALPACA_API_SECRET as env variable!")

# Result from your agent
trade_result_json = result
# Alpaca Paper Trading Credentials
API_KEY = os.environ["ALPACA_API_KEY"]
API_SECRET = os.environ["ALPACA_API_SECRET"]
BASE_URL = 'https://paper-api.alpaca.markets' 

# Create Alpaca API object
api = tradeapi.REST(API_KEY, API_SECRET, BASE_URL, api_version='v2')

def place_trade(ticker, trade_data, qty=1):
    if trade_data["action"] not in ["BUY", "SELL"]:
        print("No trade action required.")
        return
    print(f"Placing {trade_data['action']} order for {ticker}")
    # Submit bracket order
    order = api.submit_order(
        symbol=ticker,
        qty=qty,
        side=trade_data["action"].lower(),  # 'buy' or 'sell'
        type='market',
        time_in_force='gtc',
        order_class='bracket',
        stop_loss={'stop_price': trade_data["stop_loss"]},
        take_profit={'limit_price': trade_data["take_profit"]}
    )
    print(f"Order submitted: {order.id}")
    return order

place_trade("BTC/USD", agentic_trade_decision)


No trade action required.


END OF NOTEBOOK